In [14]:
%load_ext autoreload
%autoreload 2
%reload_ext autoreload

import nest_asyncio
nest_asyncio.apply()

import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "vscode"            

import matplotlib.pyplot as plt
import matplotlib.pylab as pylab
import matplotlib.dates as mdates
plt.style.use('ggplot')
params = {'legend.fontsize': 'medium',
        'figure.figsize': (20, 12),
        'axes.labelsize': 'medium',
        'axes.titlesize':'medium',
        'xtick.labelsize':'medium',
        'ytick.labelsize':'medium'}
pylab.rcParams.update(params)

import pandas as pd
import numpy as np
import rateslib as rl
import QuantLib as ql

import time
import datetime
import pytz

NYC_tz = pytz.timezone("America/New_York") 
CHI_tz = pytz.timezone("America/Chicago") 
LDN_tz = pytz.timezone("Europe/London") 
UTC_tz = pytz.timezone("UTC") 

import sys
sys.path.append("../../../")


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [15]:
from MDP.IRSwaps.IRSwapsMDP import IRSwapsMDP
from Query.Unified.registry import UnifiedValue
from Query.Unified.UnifiedQuery import UnifiedQuery
from Query.IRSwaps.IRSwapQuery import IRSwapQuery 
from TB.IRSwapsTB import IRSwapsTB
from TB.TimeseriesBuilder import TimeseriesBuilder

from RVUtils.plt_timeseries import make_secondary_axis_plot

In [16]:
curve_mdp = IRSwapsMDP(source="BARCHART_STIRF-RL")
curve_tb = IRSwapsTB(curve_mdp, show_tqdm=True)
tb = TimeseriesBuilder()

In [17]:
ts = NYC_tz.localize(datetime.datetime(2026, 5, 22, 17, 00))
# ts = "live"
curve = "USD-SOFR-1D-Q12STIRT"

curve_handle = curve_mdp.get_pricer(request=dict(curve_name=curve, timestamp=ts))
curve_handle

RLIRSwapCurve(_rl_curve_id='USD-SOFR-1D', _rl_curve_handle=<rl.Curve:USD-SOFR-1D at 0x1ef651a5e50>, _meta_data={'timestamp': Timestamp('2026-05-22 21:00:00+0000', tz='UTC'), 'id': 'BARCHART_STIRF-RL-USD-SOFR-1D-Q12STIRT-2026-05-22 21:00:00+00:00', 'requested_curve_name': 'USD-SOFR-1D-Q12STIRT', 'curve_name': 'USD-SOFR-1D-Q12STIRT'})

In [18]:
cm_tenors = [
    "IMM_1xIMM_2",
    "IMM_2xIMM_3",
    "IMM_3xIMM_4",
    "IMM_4xIMM_5",
    "IMM_5xIMM_6",
    "IMM_6xIMM_7",
    "IMM_7xIMM_8",
    "IMM_8xIMM_9",
    "IMM_9xIMM_10",
    "IMM_10xIMM_11",
    "IMM_11xIMM_12",
    "IMM_12xIMM_13",
]
abs_tenors = []

for t in cm_tenors:
    pkg, _ = IRSwapQuery(curve=curve, tenor=t).resolve_query(ts, pricer_or_curve=curve_handle).resolve_package(pricer_or_curve=curve_handle)
    

    eff = pkg[0].__dict__["kwargs"]["effective"]   
    price = 100 - pkg[0].__dict__["kwargs"]["fixed_rate"]
    print(price) 
    abs_tenors.append(ql.IMM.code(ql.Date(eff.day,eff.month,eff.year)))

abs_tenors

99.96322500001519
99.96189999993446
99.96064999994393
99.95970000019594
99.95959999995354
99.96004999994373
99.9606500002265
99.96109999960883
99.96115000014345
99.96110000010822
99.9609999995832
99.96077500017162


['M6', 'U6', 'Z6', 'H7', 'M7', 'U7', 'Z7', 'H8', 'M8', 'U8', 'Z8', 'H9']

In [22]:
def enumerate_flies(prices_df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Enumerate all butterfly and double butterfly combinations from a sorted futures price DataFrame.
    
    Columns must be sorted front-to-back (e.g. SR3M6, SR3U6, SR3Z6, ...).
    Prices are in futures price space (e.g. 99.9632).
    
    BF (buy):  +1 leg1, -2 leg2, +1 leg3
    DF (buy):  +1 leg1, -3 leg2, +3 leg3, -1 leg4
    
    Returns two DataFrames (butterflies, double_butterflies) with a row per combo
    and a column per timestamp from the input.
    """
    cols = list(prices_df.columns)
    n = len(cols)

    bf_records = []
    for s in range(1, n):
        for i in range(n - 2 * s):
            j, k = i + s, i + 2 * s
            name = f"BF {cols[i]}-{cols[j]}-{cols[k]}"
            vals = prices_df.iloc[:, i] - 2 * prices_df.iloc[:, j] + prices_df.iloc[:, k]
            bf_records.append({"name": name, "spacing": s, "series": vals * 10_000})

    df_records = []
    for s in range(1, n):
        for i in range(n - 3 * s):
            j, k, l = i + s, i + 2 * s, i + 3 * s
            name = f"DF {cols[i]}-{cols[j]}-{cols[k]}-{cols[l]}"
            vals = (
                prices_df.iloc[:, i]
                - 3 * prices_df.iloc[:, j]
                + 3 * prices_df.iloc[:, k]
                - prices_df.iloc[:, l]
            )
            df_records.append({"name": name, "spacing": s, "series": vals * 10_000})

    def _build(records):
        if not records:
            return pd.DataFrame()
        out = pd.DataFrame({r["name"]: r["series"] for r in records})
        out.columns.name = "combo"
        return out

    return _build(bf_records), _build(df_records)

In [25]:
def enumerate_calendars(prices_df: pd.DataFrame) -> pd.DataFrame:
    """Enumerate all standard calendar spread combinations from a sorted futures price DataFrame.

    Columns must be sorted front-to-back (e.g. SR3M6, SR3U6, SR3Z6, ...).
    Prices are in futures price space (e.g. 99.9632).

    SP (buy):  +1 leg1 (nearby)  -1 leg2 (deferred)
    Spread price = Leg1 - Leg2

    Returns a DataFrame where each column is a named spread and rows match the input index.
    Values are in ticks (price diff * 10,000).
    """
    cols = list(prices_df.columns)
    n = len(cols)

    records = []
    for s in range(1, n):
        for i in range(n - s):
            j = i + s
            name = f"SP {cols[i]}-{cols[j]}"
            vals = (prices_df.iloc[:, i] - prices_df.iloc[:, j]) * 10_000
            records.append({"name": name, "spacing": s, "series": vals})

    if not records:
        return pd.DataFrame()
    out = pd.DataFrame({r["name"]: r["series"] for r in records})
    out.columns.name = "combo"
    return out

In [27]:
prices = {}
for t in cm_tenors:
    pkg, _ = (
        IRSwapQuery(curve=curve, tenor=t)
        .resolve_query(ts, pricer_or_curve=curve_handle)
        .resolve_package(pricer_or_curve=curve_handle)
    )
    eff = pkg[0].__dict__["kwargs"]["effective"]
    price = 100 - pkg[0].__dict__["kwargs"]["fixed_rate"]
    imm = ql.IMM.code(ql.Date(eff.day, eff.month, eff.year))
    prices[imm] = [price]

prices_df = pd.DataFrame(prices)

sp_all= enumerate_calendars(prices_df)
bf_all, df_all = enumerate_flies(prices_df)

print(f"=== CURVES ({sp_all.shape[1]} total) ===")
display(sp_all.T.rename(columns={0: "value_ticks"}).head(20))
print(f"=== BUTTERFLIES ({bf_all.shape[1]} total) ===")
display(bf_all.T.rename(columns={0: "value_ticks"}))
print(f"\n=== DOUBLE BUTTERFLIES ({df_all.shape[1]} total) ===")
display(df_all.T.rename(columns={0: "value_ticks"}))

=== CURVES (66 total) ===


,value_ticks
combo,
SP M6-U6,13.250001
SP U6-Z6,12.500000
SP Z6-H7,9.499997
SP H7-M7,1.000002
SP M7-U7,-4.500000
SP U7-Z7,-6.000003
SP Z7-H8,-4.499994
SP H8-M8,-0.500005
SP M8-U8,0.500000


=== BUTTERFLIES (30 total) ===


,value_ticks
combo,
BF M6-U6-Z6,0.750001
BF U6-Z6-H7,3.000002
BF Z6-H7-M7,8.499995
BF H7-M7-U7,5.500002
BF M7-U7-Z7,1.500003
BF U7-Z7-H8,-1.500009
BF Z7-H8-M8,-3.999988
BF H8-M8-U8,-1.000006
BF M8-U8-Z8,-0.500005



=== DOUBLE BUTTERFLIES (18 total) ===


,value_ticks
combo,
DF M6-U6-Z6-H7,-2.250002
DF U6-Z6-H7-M7,-5.499993
DF Z6-H7-M7-U7,2.999993
DF H7-M7-U7-Z7,3.999999
DF M7-U7-Z7-H8,3.000012
DF U7-Z7-H8-M8,2.499979
DF Z7-H8-M8-U8,-2.999983
DF H8-M8-U8-Z8,-0.500001
DF M8-U8-Z8-H9,0.749984


In [18]:
start = NYC_tz.localize(datetime.datetime(2025, 1, 10, 18, 00))
end = NYC_tz.localize(datetime.datetime(2026, 5, 18, 11, 00))

q = UnifiedQuery(
    curve="USD-SOFR-1D-Q12STIRT",
    tenor="IMM_Z26xIMM_H27",
    value=UnifiedValue.IRS_RATE,
)
# q1 = UnifiedQuery(
#     curve="USD-SOFR-1D-Q12STIRT",
#     tenor="IMM_H27xIMM_M27",
#     value=UnifiedValue.IRS_RATE,
# )
# q2 = UnifiedQuery(
#     curve="USD-SOFR-1D-Q12STIRT",
#     tenor="IMM_U27xIMM_Z27",
#     value=UnifiedValue.IRS_RATE,
# )
# q3 = UnifiedQuery(
#     curve="USD-SOFR-1D-Q12STIRT",
#     tenor="IMM_U26xIMM_Z26/IMM_H27xIMM_M27/IMM_U27xIMM_Z27",
#     value=UnifiedValue.IRS_RATE,
# )
df = ts_builder.get_timeseries(
    start=start,
    end=end,
    queries=[q,],
    freq="nyc_eod",
    n_jobs=12,
    routers={
        "IRS": IRSwapsTB(curve_mdp, show_tqdm=True),
    },
    ignore_cache_miss=True,
)
df

,USD-SOFR-1D IMM_Z26xIMM_H27 OUTRIGHT RATE
Date,
2025-01-13 17:00:00-05:00,4.199486
2025-01-14 17:00:00-05:00,4.207406
2025-01-15 17:00:00-05:00,4.037924
2025-01-16 17:00:00-05:00,3.988833
2025-01-17 17:00:00-05:00,4.045596
...,...
2026-05-11 17:00:00-04:00,3.755000
2026-05-12 17:00:00-04:00,3.800000
2026-05-13 17:00:00-04:00,3.790000


In [19]:
plot, fig, ax, ax2, legend = make_secondary_axis_plot(engine="plotly")
plot(
    df[q.col_name().replace("-Q12STIRT", "")],
    which="left",
    indicators=[
        # {"kind": "last", "style": {"linestyle": "--", "linewidth": 1.2, "color": "tab:purple"}},  
        # {"kind": "sma", "window": 60, "style": {"linestyle": "--", "color": "green"}},
    ],
    # ou={"enable": True, "steps": 126, "add_metrics_to_legend": True}
)
legend(show_date=True, loc="upper left")
plt.show()